In [12]:
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from shapely import wkt
import folium
from folium.plugins import HeatMap
import requests
import math
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve, auc, recall_score, precision_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# Configurações visuais
sns.set_theme(style="whitegrid")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../tensile-psyche-440213-g7-2317d6e23754.json"
project_id = "tensile-psyche-440213-g7"

### 9. Score de Prioridade

O Score de Prioridade ($S_p$) para um chamado $i$ é definido por:$$S_{p,i} = 0.4 \cdot \hat{P}_{atraso,i} + 0.3 \cdot \text{Proxy}_{territorio,i} + 0.2 \cdot \text{Proxy}_{impacto,i} + 0.1 \cdot \text{Proxy}_{clima,i}$$

- $\hat{P}_{atraso,i}$: Probabilidade de Atraso. Saída direta do seu modelo (1 - predict\_proba). Representa o risco operacional.
- $T_{equity,i}$: Proxy de Equidade Territorial. Peso atribuído à Subprefeitura. Áreas como "Zona Norte" e "Zona Oeste" recebem peso 1.0, enquanto "Zona Sul" recebe 0.2.
- $I_{urgent,i}$: Proxy de Impacto do Serviço. Peso baseado na coluna tipo. Serviços como "Drenagem", "Arboviroses" e "Saneamento" recebem 1.0; "Estacionamento" recebe 0.1.
- $C_{context,i}$: Contexto Climático. Valor normalizado da coluna chuva\_acum\_3d. Se a região está saturada por chuvas passadas, a prioridade aumenta.


Estratégia de Implementação: O Equilíbrio Eficiência vs. Equidade
O score resolve o dilema do gestor público ao não permitir que a eficiência logística negligencie a vulnerabilidade social:

Pilar da Eficiência: Ao atribuir 40% ao risco de SLA, o sistema identifica gargalos operacionais antes que eles se tornem crises.

Pilar da Equidade: O peso de 30% para o território atua como um "corretor de desigualdade". Em cenários de empate técnico operacional, o chamado em uma área de menor IDH ou maior risco social será priorizado automaticamente.

A Regra dos 20%: A aplicação prática segue o método de Top-K Selection. Apenas os chamados no percentil 80 (maiores scores) entram no "Fluxo de Ouro", recebendo despacho imediato e monitoramento intensivo da central de controle.

### 10. Simulação e Impacto


In [5]:
import joblib
rf_model = joblib.load("../results/random_forest_model.pkl")
print("Modelo Random Forest carregado com sucesso!")
X_test = pd.read_parquet("../data/X_test.parquet")
y_test = pd.read_parquet("../data/y_test.parquet")
print("Dados de teste carregados com sucesso!")

Modelo Random Forest carregado com sucesso!
Dados de teste carregados com sucesso!


In [10]:
if isinstance(y_test, pd.DataFrame):
    y_test = y_test.iloc[:, 0]

y_prob_atraso = 1 - rf_model.predict_proba(X_test)[:, 1]

In [11]:
colunas_vulneraveis = [
    'subprefeitura_Subprefeitura da Zona Norte',
    'subprefeitura_Subprefeitura da Zona Oeste I',
    'subprefeitura_Subprefeitura da Zona Oeste II',
    'subprefeitura_Subprefeitura da Zona Oeste III',
    'subprefeitura_Subprefeitura dos Grandes Complexos',
    'area_planejamento_3', 
    'area_planejamento_5' 
]

cols_presentes = [c for c in colunas_vulneraveis if c in X_test.columns]
proxy_territorio = X_test[cols_presentes].max(axis=1).values

proxy_categoria = X_test['categoria_SERVIÇO'].values

p_min, p_max = X_test['pressao_local_tipo'].min(), X_test['pressao_local_tipo'].max()
proxy_pressao = (X_test['pressao_local_tipo'] - p_min) / (p_max - p_min)

X_test_scored = X_test.copy()
X_test_scored['score_sp'] = (
    0.4 * y_prob_atraso + 
    0.3 * proxy_territorio + 
    0.2 * proxy_categoria + 
    0.1 * proxy_pressao
)

In [14]:
y_real_atraso = (y_test == 0).astype(int) 

# Estratégia A: Aleatória
np.random.seed(42)
idx_random = np.random.choice(len(y_test), size=int(0.2 * len(y_test)), replace=False)
preds_random = np.zeros(len(y_test))
preds_random[idx_random] = 1

# Estratégia B: Top 20% pelo Score Sp
limiar_20 = np.percentile(X_test_scored['score_sp'], 80)
preds_score = (X_test_scored['score_sp'] >= limiar_20).astype(int)

print("\n" + "="*40)
print("RESULTADOS DA SIMULAÇÃO (CAPACIDADE 20%)")
print("="*40)

for nome, preds in [("Aleatória", preds_random), ("Score Sp", preds_score)]:
    prec = precision_score(y_real_atraso, preds)
    rec = recall_score(y_real_atraso, preds)
    print(f"Estratégia {nome:<10}: Precision = {prec:.4f} | Recall = {rec:.4f}")

lift = recall_score(y_real_atraso, preds_score) / 0.20
print("-" * 40)
print(f"Ganho de Eficiência (Lift): {lift:.2f}x")
print("="*40)


RESULTADOS DA SIMULAÇÃO (CAPACIDADE 20%)
Estratégia Aleatória : Precision = 0.2568 | Recall = 0.1967
Estratégia Score Sp  : Precision = 0.4427 | Recall = 0.3391
----------------------------------------
Ganho de Eficiência (Lift): 1.70x


A simulação final no conjunto de teste de 2024 valida a eficácia do Score de Prioridade ($S_p$). Ao restringirmos a capacidade de atenção especial a 20% do volume total de chamados, comparamos a capacidade de interceptação de atrasos de cada abordagem.

O modelo saltou de uma precisão de $25\%$ para $44\%$. Isso significa que, a cada 100 chamados que o  sistema marca como "Prioridade Máxima", 44 realmente sofreriam atraso se não houvesse intervenção. No método aleatório, seriam apenas 25. Assim reduzimos drasticamente o desperdício de recursos em chamados que seriam resolvidos naturalmente. Além disso, com apenas 20% da força operacional, fomos capazes de capturar e monitorar 33,9% de todos os atrasos da cidade. É um ganho massivo de cobertura preventiva.